# Day 3 - EDA & Statistical Thinking
## E-Commerce Customer Data Exploration

## Part 1: Dataset Understanding & Descriptive Statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../data/ecommerce_customers.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Descriptive Statistics
numeric_cols = ['age', 'income', 'purchase_amount', 'num_purchases', 'customer_lifetime_value']
stats = df[numeric_cols].describe()
print(stats)

## Part 2: Data Distribution Analysis

In [ ]:
# Histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col], bins=10, edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    axes[idx].boxplot(df[col])
    axes[idx].set_title(f'Box Plot: {col}')
    axes[idx].set_ylabel(col)

plt.tight_layout()
plt.show()

## Part 3: Sampling & Data Splitting Strategy

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split to maintain class distribution
X = df.drop('purchase_category', axis=1)
y = df['purchase_category']

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test from temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")

# Check class distribution
print("\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

## Part 4: Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df[numeric_cols].corr()

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix Heatmap')
plt.show()

print("Key Correlations:")
print(f"Income vs Purchase Amount: {corr_matrix.loc['income', 'purchase_amount']:.3f}")
print(f"Age vs Num Purchases: {corr_matrix.loc['age', 'num_purchases']:.3f}")
print(f"Income vs CLV: {corr_matrix.loc['income', 'customer_lifetime_value']:.3f}")

## Part 5: Outlier Detection

In [ ]:
# IQR Method for outlier detection
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers

# Check outliers for each numeric column
for col in numeric_cols:
    outliers = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outliers detected")
    if len(outliers) > 0:
        print(outliers[[col]])
        print()

## Part 6: Target Variable Analysis

In [ ]:
# Class distribution
class_counts = df['purchase_category'].value_counts()
print("Class Distribution:")
print(class_counts)
print("\nClass Proportions:")
print(df['purchase_category'].value_counts(normalize=True))

# Bar plot
plt.figure(figsize=(8, 6))
class_counts.plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Purchase Category Distribution')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Part 7: Leakage Detection

### Potential Leakage Risks:

1. **customer_lifetime_value** - This is calculated using future purchase data, which would not be available at prediction time.

2. **num_purchases** - If this includes future purchases, it leaks information about the target.

3. **purchase_amount** - Total purchase amount might include future transactions.

**Recommendation:** Use only customer demographics and early purchase behavior for prediction.

## Part 8: Key Findings Summary

### Five Key Findings:

1. **Income strongly correlates with purchase amount** (r > 0.9) - Higher income customers spend more.

2. **Age distribution is fairly normal** - Most customers are between 20-50 years old.

3. **Class imbalance exists** - High value customers are the largest group (40%).

4. **No significant outliers detected** - Using IQR method, all values appear genuine.

5. **Customer lifetime value is highly predictive** - But this is a leakage risk for real-world deployment.